# Matched Filter for Gravitational Wave Detection — PyCBC

This notebook implements the same **matched filter** analysis as `matched_filter.ipynb` (which uses [ripple](https://github.com/tedwards2412/ripple)), but here we use [PyCBC](https://pycbc.org/) — the production gravitational-wave analysis library.

PyCBC provides high-level, optimised implementations of every step:
- `pycbc.psd` — built-in detector PSDs
- `pycbc.noise` — coloured Gaussian noise generation
- `pycbc.waveform` — large catalogue of waveform approximants
- `pycbc.filter` — matched filter, overlap, and match functions

## Signal model

We study a Binary Neutron Star (BNS) merger using the **TaylorF2** frequency-domain post-Newtonian approximant, matching the parameters used in the ripple notebook for direct comparison.

## Outline

1. Setup — PSD, segment parameters
2. Waveform generation — `get_fd_waveform` with TaylorF2
3. Noise + injection — `noise_from_psd`, SNR-scaled injection
4. Matched filter — `pycbc.filter.matched_filter`
5. Template bank search — grid over $(m_1, m_2)$
6. Match and fitting factor — `pycbc.filter.match`
7. Power $\chi^2$ veto — sub-template consistency test
8. Comparison with ripple results

## 1. Imports and Setup

In [ ]:
%config InlineBackend.figure_format = 'retina'

import numpy as np
import matplotlib.pyplot as plt

import pycbc.noise
import pycbc.psd
import pycbc.waveform
import pycbc.filter
import pycbc.vetoes
from pycbc.types import FrequencySeries, TimeSeries

print(f"PyCBC version : {pycbc.__version__}")

In [ ]:
# ── Source parameters (same as matched_filter.ipynb for comparison) ───────────
m1_true  = 1.4    # Solar masses
m2_true  = 1.3    # Solar masses
chi1_z   = 0.02   # Aligned spin on body 1
chi2_z   = 0.01   # Aligned spin on body 2
dist_mpc = 100.0  # Luminosity distance [Mpc]
lambda1  = 300.0  # Tidal deformability body 1
lambda2  = 300.0  # Tidal deformability body 2

# ── Frequency / time parameters ───────────────────────────────────────────────
f_low       = 20.0          # Hz — low-frequency cutoff
sample_rate = 2048          # Hz — sampling rate
T           = 128.0         # s  — segment duration (must fit full BNS inspiral)

delta_t = 1.0 / sample_rate
delta_f = 1.0 / T
N       = int(T * sample_rate)   # total samples
flen    = N // 2 + 1             # rfft length

target_snr = 12.0   # Injection SNR

print(f"Segment duration : {T:.0f} s")
print(f"Sample rate      : {sample_rate} Hz")
print(f"delta_f          : {delta_f:.4f} Hz")
print(f"N samples        : {N}")

## 2. Noise PSD

PyCBC ships with tabulated and analytic PSDs. We use `aLIGOZeroDetHighPower` — the Advanced LIGO design sensitivity (zero-detuning, high-power) — which is also the curve used in the ripple notebook.

In [ ]:
# aLIGOZeroDetHighPower returns a FrequencySeries of length flen
psd = pycbc.psd.aLIGOZeroDetHighPower(flen, delta_f, f_low)

print(f"PSD length  : {len(psd)} bins")
print(f"PSD delta_f : {psd.delta_f:.4f} Hz")
print(f"PSD f_low   : {psd.sample_frequencies[psd.data > 0][0]:.1f} Hz")

# Plot
freqs = np.array(psd.sample_frequencies)
psd_vals = np.array(psd)
mask = psd_vals > 0

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(freqs[mask], np.sqrt(psd_vals[mask]), color='steelblue')
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel(r'$\sqrt{S_n(f)}$ [Hz$^{-1/2}$]')
ax.set_title('aLIGO Zero-Det High-Power PSD (PyCBC built-in)')
ax.set_xlim(f_low, sample_rate / 2)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Waveform Generation with PyCBC

`pycbc.waveform.get_fd_waveform` returns a `FrequencySeries` for both the $+$ and $\times$ polarisations. For a face-on, optimally oriented source we use $h_+$ only.

In [ ]:
hp_signal, hc_signal = pycbc.waveform.get_fd_waveform(
    approximant = 'TaylorF2',
    mass1       = m1_true,
    mass2       = m2_true,
    spin1z      = chi1_z,
    spin2z      = chi2_z,
    distance    = dist_mpc,
    lambda1     = lambda1,
    lambda2     = lambda2,
    f_lower     = f_low,
    f_final     = sample_rate / 2.0,
    delta_f     = delta_f,
)

print(f"Waveform length : {len(hp_signal)} bins (before resize)")
print(f"Waveform delta_f: {hp_signal.delta_f:.4f} Hz")

# Resize to match the PSD / data length
hp_signal.resize(flen)

# Compute the optimal SNR at the specified distance
opt_snr = pycbc.filter.sigma(
    hp_signal, psd=psd, low_frequency_cutoff=f_low
)
print(f"Optimal SNR at {dist_mpc} Mpc : {opt_snr:.2f}")

In [ ]:
# Plot waveform amplitude vs frequency
hp_arr   = np.array(hp_signal)
f_arr    = np.array(hp_signal.sample_frequencies)
nonzero  = np.abs(hp_arr) > 0

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].loglog(f_arr[nonzero], np.abs(hp_arr[nonzero]),
               color='steelblue', label=r'$|\tilde{h}_+(f)|$')
axes[0].loglog(freqs[mask], np.sqrt(psd_vals[mask]) / opt_snr,
               '--', color='tomato', alpha=0.7,
               label=r'$\sqrt{S_n(f)}\,/\,\rho_{\rm opt}$')
axes[0].set_ylabel('Strain [Hz$^{-1}$]')
axes[0].legend()
axes[0].grid(True, which='both', alpha=0.3)
axes[0].set_title(
    f'TaylorF2 BNS — $m_1={m1_true}\,M_\\odot$, $m_2={m2_true}\,M_\\odot$, $d_L={dist_mpc}$ Mpc'
)

axes[1].semilogx(f_arr[nonzero], np.unwrap(np.angle(hp_arr[nonzero])), color='steelblue')
axes[1].set_xlabel('Frequency [Hz]')
axes[1].set_ylabel('Phase [rad]')
axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Generate Noise and Inject Signal

`pycbc.noise.noise_from_psd` draws coloured Gaussian noise whose one-sided PSD matches `psd`. We then scale the waveform to achieve a desired injection SNR and inject it in the frequency domain.

In [ ]:
# ── Generate coloured Gaussian noise ──────────────────────────────────────────
strain_noise = pycbc.noise.noise_from_psd(N, delta_t, psd, seed=42)
print(f"Noise duration : {strain_noise.duration:.1f} s")
print(f"Noise delta_t  : {strain_noise.delta_t:.6f} s")

# Convert to frequency domain
stilde_noise = strain_noise.to_frequencyseries()

# ── Scale injection to target SNR ─────────────────────────────────────────────
# sigma(h) = sqrt(<h|h>) — the template normalisation
sig_h = pycbc.filter.sigma(hp_signal, psd=psd, low_frequency_cutoff=f_low)
scale = target_snr / sig_h

hp_injection = hp_signal * scale   # FrequencySeries multiplication

# Verify
sig_inj = pycbc.filter.sigma(hp_injection, psd=psd, low_frequency_cutoff=f_low)
print(f"Injection SNR  : {sig_inj:.2f}  (target: {target_snr})")

# ── Inject in frequency domain ────────────────────────────────────────────────
stilde_data = stilde_noise.copy()
stilde_data += hp_injection    # Adds signal at its default coalescence time

# Back to time domain for visualisation
strain_data  = stilde_data.to_timeseries()
strain_signal_only = hp_injection.to_timeseries()

In [ ]:
# ── Find the coalescence time (peak of |h(t)|) ────────────────────────────────
sig_t  = np.array(strain_signal_only)
t_sig  = np.array(strain_signal_only.sample_times)
tc_idx = np.argmax(np.abs(sig_t))
tc_pycbc = t_sig[tc_idx]
print(f"Coalescence time in segment: t = {tc_pycbc:.4f} s")

# Plot ±0.5 s around coalescence
t_data = np.array(strain_data.sample_times)
d_data = np.array(strain_data)

window_s = 0.5
mask_t = np.abs(t_data - tc_pycbc) < window_s

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_data[mask_t], d_data[mask_t],
        lw=0.5, color='grey', alpha=0.7, label='Data (signal + noise)')
ax.plot(t_sig[mask_t], sig_t[mask_t],
        lw=1.2, color='steelblue', label=f'Injected signal (SNR = {target_snr})')
ax.axvline(tc_pycbc, color='tomato', ls='--', lw=1, alpha=0.8, label=f'$t_c$ = {tc_pycbc:.3f} s')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Strain')
ax.set_title('BNS injection — PyCBC')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Matched Filter

`pycbc.filter.matched_filter` returns the **complex** SNR time series $z(t)$ normalised so that $|z(t)| = \rho(t)$. It handles all frequency-domain weighting, normalisation, and the inverse FFT internally.

We crop the edges to remove artifacts from the PSD estimation filter (±4 s) and from the template wrapping (one template duration).

In [ ]:
# ── Matched filter with exact (same-parameters) template ─────────────────────
# Use the unscaled template — pycbc normalises by sigma internally
snr_exact = pycbc.filter.matched_filter(
    hp_signal,
    stilde_data,
    psd              = psd,
    low_frequency_cutoff  = f_low,
    high_frequency_cutoff = sample_rate / 2.0,
)

# Estimate template duration (time between f_low and merger)
# For BNS / TaylorF2 the number of non-zero bins gives us the fd waveform length
hp_nonzero_bins = np.sum(np.abs(np.array(hp_signal)) > 0)
template_duration = hp_nonzero_bins * delta_f  # approximate
print(f"Approximate template duration : {template_duration:.1f} s")

# Crop to remove edge effects
# Start: template duration + 4 s (PSD filter); End: 4 s
crop_start = min(template_duration + 4.0, T / 2)
crop_end   = 4.0
snr_cropped = snr_exact.crop(crop_start, crop_end)

snr_t_arr = np.abs(np.array(snr_cropped))
t_snr_arr = np.array(snr_cropped.sample_times)

# Peak
peak_idx  = np.argmax(snr_t_arr)
peak_snr  = snr_t_arr[peak_idx]
peak_time = t_snr_arr[peak_idx]

print(f"Peak SNR        : {peak_snr:.2f}  (injected: {target_snr})")
print(f"Peak time       : {peak_time:.4f} s  (true tc: {tc_pycbc:.4f} s)")

In [ ]:
# Plot the SNR time series around the peak
plot_window = 1.0
mask_snr = np.abs(t_snr_arr - tc_pycbc) < plot_window

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_snr_arr[mask_snr], snr_t_arr[mask_snr],
        lw=1.0, color='steelblue', label='Matched filter SNR')
ax.axvline(tc_pycbc, color='tomato', ls='--', lw=1.5,
           label=f'True $t_c$ = {tc_pycbc:.4f} s')
ax.axvline(peak_time, color='green', ls=':', lw=1.5,
           label=f'Recovered $t_c$ = {peak_time:.4f} s')
ax.axhline(target_snr, color='grey', ls='-.', alpha=0.6,
           label=f'Injection SNR = {target_snr}')
ax.set_xlabel('Time [s]')
ax.set_ylabel('SNR')
ax.set_title('Matched Filter SNR — PyCBC (exact template)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Template Bank Search

We scan a grid of $(m_1, m_2)$ templates, running `matched_filter` for each and recording the peak SNR inside a search window around the known coalescence time.

In [ ]:
m1_grid = np.linspace(1.1, 1.8, 10)   # Solar masses
m2_grid = np.linspace(1.1, 1.7, 9)    # Solar masses

bank_results = []   # (m1, m2, peak_snr, peak_time)

for m1_t in m1_grid:
    for m2_t in m2_grid:
        if m2_t > m1_t:
            continue

        hp_t, _ = pycbc.waveform.get_fd_waveform(
            approximant = 'TaylorF2',
            mass1       = m1_t,
            mass2       = m2_t,
            spin1z      = chi1_z,
            spin2z      = chi2_z,
            distance    = dist_mpc,
            lambda1     = lambda1,
            lambda2     = lambda2,
            f_lower     = f_low,
            f_final     = sample_rate / 2.0,
            delta_f     = delta_f,
        )
        hp_t.resize(flen)

        snr_t = pycbc.filter.matched_filter(
            hp_t, stilde_data, psd=psd,
            low_frequency_cutoff=f_low,
            high_frequency_cutoff=sample_rate / 2.0,
        )

        # Crop edges
        try:
            snr_t_crop = snr_t.crop(crop_start, crop_end)
        except Exception:
            snr_t_crop = snr_t

        snr_arr_t = np.abs(np.array(snr_t_crop))
        t_arr_t   = np.array(snr_t_crop.sample_times)

        # Search within ±0.5 s of known tc
        search_mask = np.abs(t_arr_t - tc_pycbc) < 0.5
        if search_mask.sum() == 0:
            continue

        local_peak = snr_arr_t[search_mask].max()
        local_time = t_arr_t[search_mask][np.argmax(snr_arr_t[search_mask])]
        bank_results.append((m1_t, m2_t, float(local_peak), float(local_time)))

bank_results = np.array(bank_results)
best_idx = np.argmax(bank_results[:, 2])
best = bank_results[best_idx]

print(f"Best template  : m1 = {best[0]:.2f}, m2 = {best[1]:.2f} M_sun")
print(f"Best SNR       : {best[2]:.2f}")
print(f"Recovered time : {best[3]:.4f} s  (true: {tc_pycbc:.4f} s)")
print(f"True params    : m1 = {m1_true:.2f}, m2 = {m2_true:.2f} M_sun")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(bank_results[:, 0], bank_results[:, 1],
                c=bank_results[:, 2], cmap='viridis', s=80,
                edgecolors='k', lw=0.5)
plt.colorbar(sc, ax=ax, label='Peak SNR')
ax.scatter(m1_true, m2_true, marker='*', s=300, color='tomato',
           zorder=5, label='True parameters')
ax.scatter(best[0], best[1], marker='D', s=120, color='lime',
           zorder=4, label='Best template')
ax.set_xlabel(r'$m_1$ [$M_\odot$]')
ax.set_ylabel(r'$m_2$ [$M_\odot$]')
ax.set_title('Template Bank SNR Map — PyCBC')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Match and Fitting Factor

`pycbc.filter.match` computes the overlap between two waveforms, maximised over time and phase. The **fitting factor** is the best match between the injected signal and any template in the bank — it quantifies how much SNR is lost due to parameter discretisation.

In [ ]:
matches = []

for row in bank_results:
    m1_t, m2_t = row[0], row[1]

    hp_t, _ = pycbc.waveform.get_fd_waveform(
        approximant = 'TaylorF2',
        mass1       = m1_t,
        mass2       = m2_t,
        spin1z      = chi1_z,
        spin2z      = chi2_z,
        distance    = dist_mpc,
        lambda1     = lambda1,
        lambda2     = lambda2,
        f_lower     = f_low,
        f_final     = sample_rate / 2.0,
        delta_f     = delta_f,
    )

    # match() requires equal-length arrays — pad to the longer one
    tlen = max(len(hp_signal), len(hp_t))
    hp_signal_r = hp_signal.copy()
    hp_signal_r.resize(tlen)
    hp_t.resize(tlen)
    psd_r = pycbc.psd.aLIGOZeroDetHighPower(tlen // 2 + 1, delta_f, f_low)

    m, _ = pycbc.filter.match(
        hp_signal_r, hp_t,
        psd                  = psd_r,
        low_frequency_cutoff = f_low,
    )
    matches.append(m)

matches    = np.array(matches)
best_match = matches.max()
best_m_idx = matches.argmax()

print(f"Fitting factor (best match) : {best_match:.4f}")
print(f"Best-match template         : m1 = {bank_results[best_m_idx, 0]:.2f}, "
      f"m2 = {bank_results[best_m_idx, 1]:.2f} M_sun")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(bank_results[:, 0], bank_results[:, 1],
                c=matches, cmap='plasma', vmin=0.8, vmax=1.0,
                s=80, edgecolors='k', lw=0.5)
plt.colorbar(sc, ax=ax, label='Match')
ax.scatter(m1_true, m2_true, marker='*', s=300, color='cyan',
           zorder=5, label='True parameters')
ax.set_xlabel(r'$m_1$ [$M_\odot$]')
ax.set_ylabel(r'$m_2$ [$M_\odot$]')
ax.set_title('Template Match Map — PyCBC')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Power $\chi^2$ Veto

The **power $\chi^2$** test (Allen 2005) splits the template into $p$ equal-power sub-bands and checks whether each contributes its expected fraction of the total SNR. Noise glitches produce high SNR but fail this test because their power is concentrated in a narrow frequency band.

$$\chi^2_r = \frac{1}{2p - 2} \sum_{i=1}^p \left| \rho_i - \frac{\rho}{p} \right|^2$$

For a genuine signal, $\chi^2_r \approx 1$; for a glitch, $\chi^2_r \gg 1$.

In [ ]:
num_bins = 16   # Number of equal-power sub-bands

chisq_ts = pycbc.vetoes.power_chisq(
    hp_signal,
    stilde_data,
    num_bins,
    psd,
    low_frequency_cutoff  = f_low,
    high_frequency_cutoff = sample_rate / 2.0,
)

# Reduced chi-squared: divide by (2 * num_bins - 2) degrees of freedom
dof = 2 * num_bins - 2
chisq_r = chisq_ts / dof

# Crop to match SNR time series
try:
    chisq_r_crop = chisq_r.crop(crop_start, crop_end)
except Exception:
    chisq_r_crop = chisq_r

chisq_arr = np.array(chisq_r_crop)
t_chisq   = np.array(chisq_r_crop.sample_times)

mask_chi = np.abs(t_chisq - tc_pycbc) < plot_window

# Value at the SNR peak
tc_idx_chi = np.argmin(np.abs(t_chisq - peak_time))
print(f"Reduced chi-squared at SNR peak : {chisq_arr[tc_idx_chi]:.3f}  (expect ~1 for a signal)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(t_snr_arr[mask_snr], snr_t_arr[mask_snr],
         color='steelblue', lw=1.0, label='SNR')
ax1.axvline(tc_pycbc, color='tomato', ls='--', lw=1.2,
            label=f'True $t_c$')
ax1.axhline(target_snr, color='grey', ls='-.', alpha=0.5)
ax1.set_ylabel('SNR')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_title(r'SNR and Power $\chi^2$ Veto — PyCBC')

ax2.plot(t_chisq[mask_chi], chisq_arr[mask_chi],
         color='darkorange', lw=1.0, label=r'$\chi^2_r$')
ax2.axhline(1.0, color='grey', ls='--', alpha=0.6, label=r'$\chi^2_r = 1$ (signal)')
ax2.axvline(tc_pycbc, color='tomato', ls='--', lw=1.2)
ax2.set_xlabel('Time [s]')
ax2.set_ylabel(r'Reduced $\chi^2$')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Re-weighted SNR

PyCBC searches rank candidates by a **re-weighted SNR** that penalises high $\chi^2_r$:

$$\hat{\rho} = \begin{cases} \rho & \chi^2_r \leq 1 \\ \rho \left[\frac{1}{2}\left(1 + (\chi^2_r)^3\right)\right]^{-1/6} & \chi^2_r > 1 \end{cases}$$

This statistic strongly suppresses noise transients while leaving genuine GW signals nearly unchanged.

In [ ]:
def reweighted_snr(snr, chisq_r, index=6):
    """PyCBC-style re-weighted SNR statistic."""
    newsnr = snr.copy()
    mask   = chisq_r > 1.0
    newsnr[mask] = snr[mask] * (0.5 * (1.0 + chisq_r[mask] ** 3)) ** (-1.0 / index)
    return newsnr


# Align the two arrays (they may differ in length by a few samples)
min_len = min(len(snr_t_arr), len(chisq_arr))
snr_aligned   = snr_t_arr[:min_len]
chisq_aligned = chisq_arr[:min_len]
t_aligned     = t_snr_arr[:min_len]

newsnr = reweighted_snr(snr_aligned, chisq_aligned)

mask_rw = np.abs(t_aligned - tc_pycbc) < plot_window

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_aligned[mask_rw], snr_aligned[mask_rw],
        color='steelblue', lw=1.0, label='SNR $\\rho$', alpha=0.8)
ax.plot(t_aligned[mask_rw], newsnr[mask_rw],
        color='darkorange', lw=1.5, ls='--', label=r'Re-weighted SNR $\hat{\rho}$')
ax.axvline(tc_pycbc, color='tomato', ls='--', lw=1.2, label='True $t_c$')
ax.axhline(target_snr, color='grey', ls='-.', alpha=0.5)
ax.set_xlabel('Time [s]')
ax.set_ylabel('SNR')
ax.set_title(r'SNR vs Re-weighted SNR')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

newsnr_peak = newsnr[mask_rw].max()
print(f"Peak SNR          : {snr_aligned[mask_rw].max():.2f}")
print(f"Peak re-weighted  : {newsnr_peak:.2f}")

## 10. Comparison: PyCBC vs ripple

Both pipelines process the same BNS signal. The table below summarises the key differences.

In [ ]:
summary = {
    'Waveform library'     : ('ripplegw (TaylorF2)', 'PyCBC (TaylorF2)'),
    'Acceleration'         : ('JAX (GPU/TPU ready)', 'NumPy / FFTW (CPU)'),
    'Differentiable'       : ('Yes — jax.grad through waveform', 'No'),
    'Built-in PSDs'        : ('No (analytic formula)', 'Yes — large catalogue'),
    'Noise generation'     : ('Manual (NumPy)', 'pycbc.noise.noise_from_psd'),
    'Matched filter'       : ('Manual (IFFT)', 'pycbc.filter.matched_filter'),
    'Match function'       : ('Manual (IFFT)', 'pycbc.filter.match'),
    'Chi-squared veto'     : ('Not shown', 'pycbc.vetoes.power_chisq'),
    'Re-weighted SNR'      : ('Not shown', 'Custom (shown above)'),
    'Use case'             : ('Research / ML gradients', 'Production searches'),
}

print(f"{'Feature':<22}  {'ripple':<38}  {'PyCBC'}")
print('-' * 90)
for k, (v1, v2) in summary.items():
    print(f"{k:<22}  {v1:<38}  {v2}")

## Summary

### PyCBC highlights

* **`pycbc.psd.aLIGOZeroDetHighPower`** — analytic design PSD, no external data needed.
* **`pycbc.noise.noise_from_psd`** — correctly coloured Gaussian noise in one call.
* **`pycbc.waveform.get_fd_waveform`** — large approximant catalogue; `TaylorF2` is the standard BNS choice.
* **`pycbc.filter.matched_filter`** — returns the complex SNR time series, normalised by $\sigma_h$ internally. Edge cropping is needed to remove filter artifacts.
* **`pycbc.filter.match`** — overlap maximised over time and phase (fitting factor).
* **`pycbc.vetoes.power_chisq`** — sub-band consistency test that strongly suppresses non-Gaussian noise transients.
* **Re-weighted SNR** $\hat{\rho}$ — the standard PyCBC detection statistic used in GW searches.

### Relation to SparseBank

SparseBank aims to reduce the number of templates in a PyCBC-style bank by using the mass predictions from a neural network (Step 2) to prune templates whose chirp mass falls outside the predicted range (Step 3). The pipeline in Step 4 then runs the standard PyCBC matched filter with the reduced bank — exactly the workflow demonstrated in this notebook.